# NB 4.2 &mdash; k-NN: digues-me amb qui vas

**MP 5134** &mdash; UT4

*Dades: pingüins de l'arxipèlag Palmer.*

---
### Què farem avui

Al NB 4.1 vam fer el primer classificador amb la regressió logística. Avui en farem
servir un altre, el **k-NN** (*k nearest neighbours*, els k veïns més propers), amb la
mateixa recepta de cinc passos. La seva idea és la del refrany: *digues-me amb qui vas i
et diré qui ets*.

Com al NB 4.1, hi ha tres pauses «Ara tu» llargues per fer en grup.

Quan acabis hauries de saber respondre aquestes preguntes:

- Com decideix el k-NN de quina classe és un cas nou?
- Per què el k-NN s'equivoca tant si no escalem les variables?
- Què passa amb un k molt petit, i amb un de molt gran?
- Quan triaríeu un k-NN i quan una regressió logística?

## 1. Classificar pels veïns

El k-NN no aprèn cap fórmula. Quan entrena, només **es guarda** tots els pingüins
d'entrenament. Quan li arriba un pingüí nou, busca els k pingüins guardats que més s'hi
assemblen, els seus **veïns**, i els fa votar: l'espècie que surt més vegades és la resposta.

Provem-ho amb les tres espècies i dues columnes: la llargada del bec i la massa. Fem
servir `k=5`.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.dummy import DummyClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Còpia de les dades de palmerpenguins (Gorman et al., 2014) al repositori del mòdul
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"

VARIABLES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
COLORS = ["#0072B2", "#D55E00", "#009E73"]

df = pd.read_csv(URL_DADES).dropna(subset=VARIABLES)

COLUMNES = ["bill_length_mm", "body_mass_g"]
X = df[COLUMNES]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

logistica = LogisticRegression(max_iter=5000).fit(X_train, y_train)
referencia = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print(f"k-NN:                {knn.score(X_test, y_test):.1%}")
print(f"Regressió logística: {logistica.score(X_test, y_test):.1%}")
print(f"Referència:          {referencia.score(X_test, y_test):.1%}")

És la recepta del NB 4.1, amb un model nou: `KNeighborsClassifier`, on `n_neighbors` és
la k. El k-NN encerta un 79,1%, molt per sobre de la referència (44,2%). Però la regressió
logística, amb les mateixes dues columnes, arriba al 94,2%. Alguna cosa no va bé. Mirem
quins veïns tria per a un pingüí nou.

In [ ]:
nou = pd.DataFrame({"bill_length_mm": [47], "body_mass_g": [3700]})

distancies, posicions = knn.kneighbors(nou)
print("Decisió:", knn.predict(nou))

df.loc[X_train.index[posicions[0]], ["species"] + COLUMNES]

`kneighbors` retorna els veïns d'un pingüí, i la darrera línia els mostra com una taula.
Tots cinc pesen exactament 3.700 g, igual que el nostre, però el bec els va de 37,8 a 58 mm.
Tres són Adelie i dos Chinstrap, així que el model vota Adelie.

El problema són les unitats. Per al k-NN, la distància entre dos pingüins és una suma de
diferències, i **un gram compta igual que un mil·límetre**. Una diferència de 10 mm de bec
és enorme, però val menys que 11 g de massa, que no és res. La massa decideix tota sola i el
bec quasi no compta.

## 2. Totes les variables, a la mateixa escala

Si dibuixem la frontera de decisió, el problema es veu de seguida.

In [ ]:
DecisionBoundaryDisplay.from_estimator(
    knn, X_train, response_method="predict",
    cmap=ListedColormap(COLORS), alpha=0.2
)
sns.scatterplot(data=X_test, x="bill_length_mm", y="body_mass_g", hue=y_test,
                style=y_test, hue_order=knn.classes_, palette=COLORS)
plt.title("k-NN sense escalar")
plt.show()

Les zones són franges horitzontals: el model només mira l'altura del punt, és a dir, la
massa. La solució és la del NB 3.2: **escalar** les variables perquè totes tinguin la
mateixa importància. Ho fem amb `StandardScaler`, que fa el `fit` només amb l'entrenament.

In [ ]:
escala = StandardScaler().set_output(transform="pandas")
escala.fit(X_train)
X_train_esc = escala.transform(X_train)
X_test_esc = escala.transform(X_test)

knn_esc = KNeighborsClassifier(n_neighbors=5)
knn_esc.fit(X_train_esc, y_train)
print(f"k-NN escalat: {knn_esc.score(X_test_esc, y_test):.1%}")

nou_esc = escala.transform(nou)
distancies, posicions = knn_esc.kneighbors(nou_esc)
print("Decisió:", knn_esc.predict(nou_esc))

df.loc[X_train.index[posicions[0]], ["species"] + COLUMNES]

`set_output(transform="pandas")` fa que l'escala torni una taula amb el nom de les columnes
i no una matriu, i així els gràfics continuen funcionant. Ara el k-NN encerta un 95,3%, i
els cinc veïns del pingüí nou s'hi assemblen de debò en totes dues mesures: són cinc
Chinstrap. Fixa't que el pingüí nou **també s'ha d'escalar**, amb la mateixa `escala`.

In [ ]:
DecisionBoundaryDisplay.from_estimator(
    knn_esc, X_train_esc, response_method="predict",
    cmap=ListedColormap(COLORS), alpha=0.2
)
sns.scatterplot(data=X_test_esc, x="bill_length_mm", y="body_mass_g", hue=y_test,
                style=y_test, hue_order=knn_esc.classes_, palette=COLORS)
plt.title("k-NN escalat")
plt.show()

Ara els eixos són valors escalats: el 0 és la mitjana i cada unitat és una desviació. Les
zones ja depenen de les dues variables, i la frontera no és recta com la de la regressió
logística: el k-NN pot fer fronteres tan tortes com calgui.

**Amb el k-NN, sempre s'escala.** És un dels pocs models en què oblidar-ho fa tant de mal.

---
### Ara tu · Exercici 1: escalar o no escalar

*Uns 35 minuts, en grup.*

La funció `provar` de la cel·la següent segueix la recepta amb el model que li doneu, i
escala les columnes si `escalar=True`. Per exemple:

```python
provar(["bill_length_mm", "bill_depth_mm"], "species", KNeighborsClassifier(n_neighbors=5), escalar=False)
```

**a)** Repartiu-vos les sis parelles de variables i ompliu la taula amb el percentatge
d'encerts a la prova:

| Parella de variables | Sense escalar | Escalat |
|---|---|---|
| | | |

**b)** En quines parelles canvia molt el resultat, i en quines gens? Mireu
`df[VARIABLES].describe()` i expliqueu-ho a partir de les unitats de cada columna.

**c)** Hi ha cap parella en què escalar empitjori una mica? Vol dir que no s'ha d'escalar?

**d)** Un error molt habitual: feu `knn_esc.predict(nou)` amb el pingüí nou **sense
escalar**. Què respon? Surt cap avís? Per què és un error perillós?

**Compareu amb un altre grup.** Escriviu entre tots una regla de dues línies: quan cal
escalar, i per què.

In [ ]:
def provar(columnes, objectiu, model, escalar=True, llavor=42):
    # Segueix la recepta, escala si cal i escriu els encerts
    X_train, X_test, y_train, y_test = train_test_split(
        df[columnes], df[objectiu], test_size=0.25,
        stratify=df[objectiu], random_state=llavor
    )
    if escalar:
        escala = StandardScaler().fit(X_train)
        X_train, X_test = escala.transform(X_train), escala.transform(X_test)
    model.fit(X_train, y_train)
    print(f"{model}, {columnes}, escalat={escalar}, llavor={llavor}")
    print(f"  Entrenament: {model.score(X_train, y_train):.1%}   Prova: {model.score(X_test, y_test):.1%}")

In [ ]:
# Escriu aquí el teu codi

## 3. Quants veïns?

La k és l'únic paràmetre important del k-NN. Provem-ne uns quants valors amb les dues
columnes escalades.

In [ ]:
for k in [1, 5, 15, 51, 101, 151]:
    model_k = KNeighborsClassifier(n_neighbors=k).fit(X_train_esc, y_train)
    print(f"k={k:>3}: entrenament {model_k.score(X_train_esc, y_train):.1%}, "
          f"prova {model_k.score(X_test_esc, y_test):.1%}")

Amb `k=1`, cada pingüí d'entrenament és el seu propi veí, i per això l'entrenament
encerta el 100%: el model ha memoritzat les dades, com el polinomi de grau alt del NB 2.3.
A la prova, en canvi, es queda en un 91,9%.

A l'altre extrem, amb `k=151` voten més de la meitat dels pingüins d'entrenament. Els
Chinstrap, que només en són 51, ja no guanyen mai cap votació, i el model no en detecta
cap. Entremig hi ha una zona bona, i a la UT7 aprendrem a trobar-la amb mètode.

---
### Ara tu · Exercici 2: el k de l'equip

*Uns 35 minuts, en grup.*

**a)** Dibuixeu la frontera de `k=1`, la de `k=15` i la de `k=151`. Copieu la cel·la del
gràfic «k-NN escalat» i canvieu-hi el model. Com canvia la forma de les zones? En quin
gràfic es veu que el model ha memoritzat?

**b)** Amb `provar`, ompliu aquesta taula amb els encerts a la prova. La `llavor` canvia
quins pingüins van a la prova.

| k | llavor=42 | llavor=7 | llavor=0 |
|---|---|---|---|
| 1 | | | |
| 5 | | | |
| 15 | | | |
| 51 | | | |
| 101 | | | |

**c)** Quina k triaríeu? Fixeu-vos que la millor k no és la mateixa amb cada llavor:
trieu-ne una que vagi bé amb totes tres, i expliqueu per què és millor que triar el
número més alt d'una sola columna.

**d)** Amb la vostra k, quin percentatge de probabilitat pot donar `predict_proba`? Proveu-ho
amb un pingüí nou i expliqueu d'on surten aquests números.

**Compareu amb un altre grup.** Heu triat la mateixa k? Si no, quina defensaríeu?

In [ ]:
# Escriu aquí el teu codi

## 4. El preu del k-NN

El k-NN no fa cap feina quan entrena: només guarda les dades. La feina arriba quan
prediu, perquè ha de calcular la distància a **tots** els casos guardats. Amb 256 pingüins
no es nota, però ho podem provar amb una taula gran, copiant les dades 300 vegades.

In [ ]:
gran = pd.concat([df] * 300)
print(f"Pingüins a la taula gran: {len(gran):,}")

for nom, model_g in [("Regressió logística", LogisticRegression(max_iter=5000)),
                     ("k-NN", KNeighborsClassifier(n_neighbors=5))]:
    inici = time.perf_counter()
    model_g.fit(gran[COLUMNES], gran["species"])
    entrenar = time.perf_counter() - inici

    inici = time.perf_counter()
    model_g.predict(gran[COLUMNES].head(5000))
    predir = time.perf_counter() - inici

    print(f"{nom}: entrenar {entrenar:.3f} s, predir 5.000 pingüins {predir:.3f} s")

Els segons exactes depenen de l'ordinador, però el patró es repeteix sempre: la regressió
logística triga a entrenar i després prediu en un no res, i el k-NN entrena a l'instant
però cada predicció li costa molt més. Si un model ha de respondre milers de consultes per
segon, això importa.

| | Regressió logística | k-NN |
|---|---|---|
| Frontera | Feta de línies rectes | Tan torta com calgui |
| Escalar | Recomanable | Imprescindible |
| Entrenar | Més lent | Instantani |
| Predir | Molt ràpid | Més lent com més dades |
| Explicar una decisió | Amb el signe dels coeficients | Amb els veïns que han votat |

---
### Ara tu · Exercici 3: el duel

*Uns 40 minuts, en grup.*

L'equip de recerca torna a buscar els Chinstrap, ara amb dos models candidats. Feu servir
la columna `es_chinstrap` del NB 4.1:

```python
df["es_chinstrap"] = (df["species"] == "Chinstrap").astype(int)
```

**a)** Amb `provar`, compareu la regressió logística (`LogisticRegression(max_iter=5000)`)
amb el k-NN de la vostra k, tots dos escalats, en les sis parelles de variables. En quines
parelles guanya cada model?

**b)** Agafeu la parella del bec, `["bill_length_mm", "bill_depth_mm"]`, i dibuixeu la
frontera de tots dos models. Expliqueu, mirant els dibuixos, per què un dels dos encerta
molt més.

**c)** Trieu un model per a cada situació i justifiqueu-ho:

- **Situació A.** Una app al mòbil de l'equip ha de respondre a l'instant, i a la base de
  dades hi ha mig milió de pingüins de tota l'Antàrtida.
- **Situació B.** Un comitè científic vol entendre per què el model diu que un pingüí és
  Chinstrap abans d'acceptar-ne els resultats.

**Compareu amb un altre grup.** Heu triat el mateix model a cada situació? Hi ha cap
situació en què qualsevol dels dos serviria?

In [ ]:
# Escriu aquí el teu codi

## 5. El que ens emportem

El k-NN classifica fent votar els veïns més semblants. No té fórmula, però té dues
condicions: les variables **s'han d'escalar**, també els casos nous, i cal triar una k que no
memoritzi ni ho aplani tot. Ara ja tenim dos classificadors i un criteri per triar-ne un, que
no és només el percentatge d'encerts.

Fins ara hem comptat els errors a mà. Al NB 4.3 farem el salt a un problema real, saber si
demà plourà a Palma, i aprendrem a mesurar-los amb eines pensades per a això.

## 6. Per al debat de classe

Les plataformes de música i de sèries recomanen el que han vist els usuaris que més
s'assemblen a tu. En què s'assembla això a un k-NN? Quines «variables» creieu que fan
servir, i què passaria si una d'aquestes variables tingués una escala molt més gran que
les altres?